In [1]:
import os 

os.chdir("..")

In [2]:
from jp_imports import JPTrade
from jp_qcew import CleanQCEW
import polars as pl
import pymc as pm
import pandas as pd
from prprod_repl import ProdRepl
import arviz_base as azb
import arviz_plots as azp
azp.style.use("arviz-variat")


jt = JPTrade()
pr = ProdRepl()
cq = CleanQCEW(saving_dir="data/")

In [3]:
gdf = pr.county_geom()[["name", "geoid"]]
gdf = pl.DataFrame(gdf)
df2 = gdf.with_columns(
        pl.col("name")
            .str.normalize("NFD")
            .str.replace_all(r"[\u0300-\u036f]", "")
            .str.to_lowercase()
            .str.replace_all(" ", "_")
        )
df2 = df2.to_pandas()
df2

,name,geoid
0,bayamon,72021
1,florida,72054
2,ciales,72039
3,carolina,72031
4,villalba,72149
...,...,...
73,san_german,72125
74,maunabo,72095
75,orocovis,72107
76,san_lorenzo,72129


In [4]:
df = pr.data_qcew()
df = df.with_columns(
    name=(pl.col("phys_addr_city")
            .str.normalize("NFD")
            .str.replace_all(r"[\u0300-\u036f]", "")
            .str.to_lowercase()
            .str.replace_all("  ", "_")
            .str.replace_all(" ", "_")
        )
)
df = df.to_pandas()
df

,year,month,sector_code,qtr,phys_addr_city,employment,name
0,2009,5,56,2,TOA ALTA,50,toa_alta
1,2013,1,71,1,AGUADA,5,aguada
2,2004,7,22,3,ISABELA,54,isabela
3,2020,12,45,4,VILLALBA,15,villalba
4,2003,6,45,2,HUMACAO,1508,humacao
...,...,...,...,...,...,...,...
620782,2017,6,42,2,CAGUAS,2487,caguas
620783,2021,6,56,2,COROZAL,68,corozal
620784,2012,1,55,1,MAYAQUEZ,143,mayaquez
620785,2009,4,56,2,SKOKIE,1,skokie


In [5]:
import pandas as pd
from thefuzz import process, fuzz

# 1. Define a function to find the best match
def get_best_match(name, choices, threshold=80):
    # process.extractOne returns (match, score, index)
    match, score = process.extractOne(name, choices, scorer=fuzz.token_sort_ratio)
    
    # Only return the match if it's "good enough"
    return match if score >= threshold else None

# 2. Get the list of 'correct' names from df2
correct_names = df2['name'].tolist()

# 3. Create a new column in df that maps to the closest correct name
df['matched_name'] = df['name'].apply(lambda x: get_best_match(x, correct_names))

# 4. Merge df with df2 based on the new matched column
result = pd.merge(df, df2, left_on='matched_name', right_on='name', suffixes=('_original', '_cleaned'))

In [6]:
df_imports = jt.process_int_jp(level="naics", time_frame="monthly")
df_imports = df_imports.with_columns(
    sector_code=pl.col("naics").str.slice(0,2)
)#.rename({"qrt":"qtr"})
df_imports = df_imports.group_by(["year", "month", "sector_code"]).agg(pl.col("imports", "imports_qty", "exports","exports_qty","net_exports","net_qty").sum())


In [7]:
pr.reg_data()

year,month,sector_code,qtr,name,employment,imports,imports_qty,exports,exports_qty,net_exports,net_qty
i64,i64,str,i64,str,i64,i64,f64,i64,f64,i64,f64
2011,5,"""32""",2,"""ciales""",46,2709169857,4.8928e10,3414168459,5.6067e10,704998602,7.1393e9
2014,10,"""31""",4,"""carolina""",402,440862122,2.0909e8,175608803,2.3922e7,-265253319,-1.8517e8
2021,2,"""33""",1,"""vieques""",33,939788151,4.9449e9,776478766,6.7924e8,-163309385,-4.2656e9
2018,12,"""31""",4,"""arecibo""",715,366559978,1.7944e8,80113706,2.2139e7,-286446272,-1.5730e8
2010,4,"""21""",2,"""hormigueros""",0,25530916,5.9612e8,7000541,1.5772e7,-18530375,-5.8035e8
…,…,…,…,…,…,…,…,…,…,…,…
2023,6,"""33""",2,"""cabo_rojo""",39,1373868490,5.8099e9,1083584728,5.0228e8,-290283762,-5.3076e9
2011,3,"""32""",1,"""corozal""",134,2451129079,5.2360e10,3908338255,5.1912e10,1457209176,-4.4743e8
2011,6,"""31""",2,"""utuado""",50,377422625,1.7460e8,329389689,3.0522e7,-48032936,-1.4408e8


In [8]:
data = pl.DataFrame(result)
data = data.with_columns(name="name_cleaned")
data = data.group_by(["year", "month", "sector_code", "qtr", "name"]).agg(pl.col("employment").sum())
data.join(df_imports,on=["year", "month","sector_code"], how="inner", validate="m:1")

year,month,sector_code,qtr,name,employment,imports,imports_qty,exports,exports_qty,net_exports,net_qty
i64,i64,str,i64,str,i64,i64,f64,i64,f64,i64,f64
2021,9,"""21""",3,"""ponce""",26,101070574,5.5708e8,1308845,1.5064e8,-99761729,-4.0644e8
2013,3,"""32""",1,"""toa_alta""",80,2212711908,1.8079e10,4131289742,4.7958e10,1918577834,2.9879e10
2009,12,"""31""",4,"""humacao""",638,378509229,1.7792e8,284012415,3.5864e7,-94496814,-1.4206e8
2019,2,"""32""",1,"""arroyo""",14,1939321527,4.4744e9,4351789861,1.6497e9,2412468334,-2.8247e9
2019,5,"""33""",2,"""luquillo""",58,1022024106,3.0255e9,1012257766,6.5758e8,-9766340,-2.3679e9
…,…,…,…,…,…,…,…,…,…,…,…
2015,6,"""32""",2,"""santa_isabel""",12,1900988876,8.1340e9,3767799280,5.7659e10,1866810404,4.9525e10
2023,2,"""31""",1,"""adjuntas""",155,490114304,2.3788e8,105196216,2.8599e7,-384918088,-2.0928e8
2016,9,"""11""",3,"""juana_diaz""",172,57061717,8.9238e7,3925708,1.2805e6,-53136009,-8.7958e7
